# Flipkart Realme vs Samsung - 2026 refresh
Class names below come from the Flipkart HTML you sent (search results page, list layout). If Flipkart changes them again, only the `CLS` dictionary needs editing.

In [1]:
import re, time, random
import numpy as np
import pandas as pd
from datetime import date
from selenium import webdriver
from bs4 import BeautifulSoup

## 1. Class names + parsing helpers

In [2]:
# Class names taken from the page HTML (Flipkart search results, list layout)
CLS = {
    "card":   "ZFwe0M",   # <div class="ZFwe0M row">  = one phone (title, rating, specs, price)
    "title":  "RG5Slk",   # Realme N6 5G (Midnight Green, 128 GB)
    "rating": "MKiFS6",   # 4.1
    "rr":     "PvbNMB",   # 888 Ratings & 88 Reviews
    "specs":  "DTBslk",   # <li> spec lines (RAM|ROM, Display, Camera, Battery, ...)
    "price":  "hZ3P6w",   # Rs 23,490  (selling price)
    "mrp":    "kRYCnD",   # Rs 30,999  (struck-through original price)
    "disc":   "HQe8jr",   # 24% off
}

def txt(card, tag, cls, sep=" "):
    """Text of the first <tag class=cls> inside card, or NaN."""
    el = card.find(tag, class_=cls)
    return el.get_text(sep, strip=True) if el else np.nan

def find_image(card):
    """The phone photo sits just OUTSIDE the row, so climb until we find it
    (stop if the ancestor would contain more than one phone)."""
    el = card
    for _ in range(6):
        el = el.parent
        if el is None or len(el.find_all("div", class_=CLS["card"])) != 1:
            break
        img = el.find("img", src=re.compile("rukminim"))
        if img:
            return img["src"]
    return np.nan

def parse_page(soup):
    rows = []
    for card in soup.find_all("div", class_=CLS["card"]):
        specs = [li.get_text(" ", strip=True) for li in card.find_all("li", class_=CLS["specs"])]
        if not specs:                       # not a phone card (banner / accessory)
            continue
        # match each spec by keyword, NOT by position (the order and count vary per phone)
        pick = lambda *kws: next((s for s in specs if any(k in s for k in kws)), np.nan)
        rows.append({
            "Model":                        txt(card, "div", CLS["title"]),
            "Number of rating and reviews": txt(card, "span", CLS["rr"]),
            "original_price":               txt(card, "div", CLS["mrp"], sep=""),
            "Discounted_Price":             txt(card, "div", CLS["price"]),
            "Discount":                     txt(card, "div", CLS["disc"]),
            "Rating":                       txt(card, "div", CLS["rating"]),
            "RAM_ROM":                      pick("RAM", "ROM"),
            "Display":                      pick("Display"),
            "Camera":                       pick("Camera"),
            "Battery":                      pick("mAh"),
            "product_image":                find_image(card),
        })
    return rows

## 2. Open Chrome and test ONE page first
A Chrome window opens (no chromedriver path needed). Don't close or minimise it.

In [3]:
from selenium.common.exceptions import WebDriverException

def start_driver():
    """(Re)start Chrome. Called again automatically if the window is closed or crashes."""
    global driver
    try:
        driver.quit()
    except Exception:
        pass
    driver = webdriver.Chrome()

start_driver()

def load(url, wait=(4, 6), tries=3):
    for attempt in range(tries):
        try:
            driver.get(url)
            time.sleep(random.uniform(*wait))
            for y in (800, 1600, 2400, 3200, 4000):     # scroll so lazy-loaded cards/images render
                driver.execute_script(f"window.scrollTo(0, {y});")
                time.sleep(0.4)
            return BeautifulSoup(driver.page_source, "html.parser")
        except WebDriverException as e:
            print(f"  Chrome problem ({type(e).__name__}) - restarting it, try {attempt + 1}/{tries}")
            start_driver()
    raise RuntimeError("Chrome kept failing - see the error above")

soup = load("https://www.flipkart.com/search?q=realme+mobile&page=1")
dff = pd.DataFrame(parse_page(soup))
print(len(dff), "phones on page 1 | page title:", driver.title)
dff

24 phones on page 1 | page title: Realme Mobile- Buy Products Online at Best Price in India - All Categories | Flipkart.com


,Model,Number of rating and reviews,original_price,Discounted_Price,Discount,Rating,RAM_ROM,Display,Camera,Battery,product_image
0,"realme Narzo 80 Lite (Beach Gold, 64 GB)","3,546 Ratings & 189 Reviews","₹16,999","₹13,696",19% off,4.2,4 GB RAM | 64 GB ROM,17.12 cm (6.74 inch) HD+ Display,32MP Rear Camera,6300 mAh Battery,https://rukminim2.flixcart.com/image/312/312/x...
1,"realme Narzo 80 Lite (Obsidian Black, 64 GB)","3,546 Ratings & 189 Reviews","₹16,999","₹13,799",18% off,4.2,4 GB RAM | 64 GB ROM,17.12 cm (6.74 inch) HD+ Display,32MP Rear Camera,6300 mAh Battery,https://rukminim2.flixcart.com/image/312/312/x...
2,"realme Narzo 80 Lite (See Blue, 64 GB)","3,546 Ratings & 189 Reviews","₹16,999","₹13,802",18% off,4.2,4 GB RAM | 64 GB ROM,17.09 cm (6.73 inch) Display,13MP Rear Camera,6300 mAh Battery,https://rukminim2.flixcart.com/image/312/312/x...
3,"realme Narzo 100x 5g (Flash Orange, 128 GB)",103 Ratings & 11 Reviews,"₹31,999","₹19,970",37% off,4.1,4 GB RAM | 128 GB ROM,17.3 cm (6.81 inch) HD+ Display,NaN,8000 mAh Battery,https://rukminim2.flixcart.com/image/312/312/x...
4,"realme Narzo 100 Lite 5G (Frost Silver, 128 GB)","1,510 Ratings & 74 Reviews","₹27,999","₹17,977",35% off,4.2,4 GB RAM | 128 GB ROM,17.27 cm (6.8 inch) Display,13MP Rear Camera,7000 mAh Battery,https://rukminim2.flixcart.com/image/312/312/x...
5,"realme 15 5G (Silk Pink, 256 GB)","11,383 Ratings & 1,225 Reviews",NaN,"₹31,999",NaN,4.4,8 GB RAM | 256 GB ROM,17.27 cm (6.8 inch) Full HD+ Display,50MP + 8MP | 50MP Front Camera,7000 mAh Battery,https://rukminim2.flixcart.com/image/312/312/x...
6,"realme P3x 5G (Midnight Blue, 128 GB)","21,553 Ratings & 1,173 Reviews","₹17,999","₹13,999",22% off,4.3,8 GB RAM | 128 GB ROM,17.07 cm (6.72 inch) Full HD Display,50MP Rear Camera | 8MP Front Camera,6000 mAh Battery,https://rukminim2.flixcart.com/image/312/312/x...
7,"realme P3x 5G (Midnight Blue, 128 GB)","60,237 Ratings & 3,290 Reviews","₹16,999","₹12,999",23% off,4.3,6 GB RAM | 128 GB ROM,17.07 cm (6.72 inch) Full HD Display,50MP Rear Camera | 8MP Front Camera,6000 mAh Battery,https://rukminim2.flixcart.com/image/312/312/x...
8,"realme P4 Lite (Sea Blue, 64 GB)","6,977 Ratings & 459 Reviews","₹24,999","₹16,499",34% off,3.9,4 GB RAM | 64 GB ROM,17.14 cm (6.75 inch) HD+ Display,13MP Rear Camera | 5MP Front Camera,6300 mAh Battery,https://rukminim2.flixcart.com/image/312/312/x...
9,"realme P4 Lite (Beach Gold, 64 GB)","6,977 Ratings & 459 Reviews","₹24,999","₹16,499",34% off,3.9,4 GB RAM | 64 GB ROM,17.14 cm (6.75 inch) HD+ Display,13MP Rear Camera | 5MP Front Camera,6300 mAh Battery,https://rukminim2.flixcart.com/image/312/312/x...


Check `dff` above: every column should be filled (a blank `original_price` / `Discount` is fine for phones with no discount). If it shows 0 phones, tell me and send the saved page - see the next cell.

In [4]:
if dff.empty:
    open("debug_page1.html", "w", encoding="utf-8").write(driver.page_source)
    print("0 phones found. Saved debug_page1.html. Title:", driver.title, "| URL:", driver.current_url)

## 3. Scrape all pages for both brands
The result totals on Flipkart include other brands and unrelated items, so the real phone counts are much smaller. Other-brand cards are skipped, and scraping stops after 4 pages in a row with none of this brand.

In [5]:
store = {}   # phones scraped so far, kept here so nothing is lost if a run is interrupted

def scrape_brand(brand, url, start_page=1, max_pages=80, empty_limit=4):
    """url must contain {page}. Keeps only phone cards whose title contains the brand name.
    Flipkart pads search results with other brands, so we stop only after `empty_limit`
    pages IN A ROW with no phones of this brand."""
    rows = store.setdefault(brand, [])
    empty, other = 0, 0
    for page in range(start_page, max_pages + 1):
        soup = load(url.format(page=page))
        cards = parse_page(soup)
        mine = [r for r in cards if brand.lower() in str(r["Model"]).lower()]
        other += len(cards) - len(mine)
        print(f"[{brand}] page {page}: {len(mine)} {brand} phones ({len(cards) - len(mine)} other-brand skipped) | total {len(rows) + len(mine)}")
        if mine:
            rows += mine
            empty = 0
        else:
            empty += 1
            if empty >= empty_limit:
                print(f"[{brand}] no {brand} phones for {empty_limit} pages in a row -> done")
                break
    return pd.DataFrame(rows)

# TIP: for cleaner results, tick the BRAND filter on Flipkart, copy the URL from the address bar,
# and replace page=1 with page={page}. Then use that URL here instead.
REALME_URL = "https://www.flipkart.com/search?q=realme+mobile&page={page}"
SAMSUNG_URL = "https://www.flipkart.com/search?q=samsung+phone&page={page}"

realme_raw = scrape_brand("Realme", REALME_URL)
realme_raw.to_csv("realme_raw_2026.csv", index=False)
len(realme_raw)

[Realme] page 1: 24 Realme phones (0 other-brand skipped) | total 24
[Realme] page 2: 24 Realme phones (0 other-brand skipped) | total 48
[Realme] page 3: 24 Realme phones (0 other-brand skipped) | total 72
[Realme] page 4: 24 Realme phones (0 other-brand skipped) | total 96
[Realme] page 5: 24 Realme phones (0 other-brand skipped) | total 120
[Realme] page 6: 24 Realme phones (0 other-brand skipped) | total 144
[Realme] page 7: 24 Realme phones (0 other-brand skipped) | total 168
[Realme] page 8: 24 Realme phones (0 other-brand skipped) | total 192
[Realme] page 9: 24 Realme phones (0 other-brand skipped) | total 216
[Realme] page 10: 24 Realme phones (0 other-brand skipped) | total 240
[Realme] page 11: 24 Realme phones (0 other-brand skipped) | total 264
[Realme] page 12: 24 Realme phones (0 other-brand skipped) | total 288
[Realme] page 13: 24 Realme phones (0 other-brand skipped) | total 312
[Realme] page 14: 24 Realme phones (0 other-brand skipped) | total 336
[Realme] page 15: 2

665

If a run is ever interrupted, your phones so far are still in `store['Realme']`. Continue with `scrape_brand('Realme', REALME_URL, start_page=N+1)` instead of starting over.

In [ ]:
samsung_raw = scrape_brand("Samsung", SAMSUNG_URL)
samsung_raw.to_csv("samsung_raw_2026.csv", index=False)
len(samsung_raw)

In [6]:
driver.quit()   # close Chrome

## 4. Cleaning (same columns as your old `Mobile_Phones.csv`, plus `scraped_on`)

In [10]:
def num(s):
    """'Rs 1,85,999' -> 185999.0 ; NaN-safe"""
    if pd.isna(s):
        return np.nan
    m = re.sub(r"[^\d.]", "", str(s))
    return float(m) if m else np.nan

def clean(raw, brand):
    df = raw.copy()
    df["brand"] = brand
    df["Discounted_Price"] = df["Discounted_Price"].map(num)
    df["original_price"]   = df["original_price"].map(num)
    df = df.dropna(subset=["Discounted_Price"])
    df["original_price"] = df["original_price"].fillna(df["Discounted_Price"])   # no discount -> same price

    d = df["Discount"].astype(str).str.extract(r"(\d+)\s*%")[0].astype(float)
    calc = ((df["original_price"] - df["Discounted_Price"]) / df["original_price"] * 100).round()
    df["Discount"] = d.fillna(calc).fillna(0).astype(int)

    rr = df["Number of rating and reviews"].astype(str)
    df["No_of_Ratings"] = rr.str.extract(r"([\d,]+)\s*Ratings?")[0].str.replace(",", "").astype(float)
    df["No_of_Reviews"] = rr.str.extract(r"([\d,]+)\s*Reviews?")[0].str.replace(",", "").astype(float)
    df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")

    df["Battery_in_mAh"]  = df["Battery"].astype(str).str.extract(r"(\d+)\s*mAh")[0].astype(float)
    df["Display_size_cm"] = df["Display"].astype(str).str.extract(r"([\d.]+)\s*cm")[0].astype(float)
    mp = df["Camera"].astype(str).str.extract(r"(\d+(?:\.\d+)?)\s*MP")[0]     # first (rear) camera
    df["Camera"] = mp + "MP"
    df["RAM"] = df["RAM_ROM"].astype(str).str.extract(r"(\d+\s*(?:GB|MB))\s*RAM")[0]
    df["ROM"] = df["RAM_ROM"].astype(str).str.extract(r"(\d+\s*(?:GB|TB|MB))\s*ROM")[0]

    m = df["Model"].astype(str)
    df["model"] = m.str.split("(").str[0].str.strip()
    df["Color"] = m.str.extract(r"\(([^,)]+)")[0].str.strip()

    cols = ["model", "brand", "Color", "RAM", "ROM", "original_price", "Discount", "Discounted_Price",
            "Display_size_cm", "Battery_in_mAh", "Camera", "Rating", "No_of_Ratings", "No_of_Reviews",
            "product_image"]
    out = df[cols].drop_duplicates(subset=cols[:-1]).reset_index(drop=True)
    out["original_price"]   = out["original_price"].astype(int)
    out["Discounted_Price"] = out["Discounted_Price"].astype(int)
    out["scraped_on"] = date.today().isoformat()
    return out

realme = clean(realme_raw, "Realme")
samsung = clean(samsung_raw, "SAMSUNG")
phones  = pd.concat([realme, samsung], ignore_index=True)
phones.to_csv("Mobile_Phones_2026.csv", index=False)
phones.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1502 entries, 0 to 1501
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   model             1502 non-null   object 
 1   brand             1502 non-null   object 
 2   Color             1502 non-null   object 
 3   RAM               1502 non-null   object 
 4   ROM               1501 non-null   object 
 5   original_price    1502 non-null   int64  
 6   Discount          1502 non-null   int64  
 7   Discounted_Price  1502 non-null   int64  
 8   Display_size_cm   1502 non-null   float64
 9   Battery_in_mAh    1499 non-null   float64
 10  Camera            1467 non-null   object 
 11  Rating            1484 non-null   float64
 12  No_of_Ratings     1484 non-null   float64
 13  No_of_Reviews     1484 non-null   float64
 14  product_image     1502 non-null   object 
 15  scraped_on        1502 non-null   object 
dtypes: float64(5), int64(3), object(8)
memory 